# Reconstructing SegRNN — from the paper and the original code only

An independent reconstruction of the paper's Table II (multivariate) and
Table III (univariate) for ETTh1, ETTh2, ETTm1 and ETTm2.

Two sources are used, and nothing else:

1. **The paper**, for the published values being reproduced.
2. **The authors' original repository**, `lss-1138/SegRNN`, pinned to
   commit `8e869ec` — the last upstream commit, before any change made
   for this project.

**Why this exists separately.** This project's fork modifies
`run_longExp.py`, `exp/exp_main.py`, `data_provider/` and `utils/tools.py`
to support its Stage 2 experiments. Those changes are additive and
gated behind flags that default to off, so a plain `SegRNN` run should be
unaffected — but "should be" is an assumption, and this notebook removes
the need to make it. It runs on code that has never been touched.

**Hyperparameters are not restated here.** This notebook does not
re-declare the settings in a Python dictionary. It executes the
repository's own shell scripts verbatim — `scripts/SegRNN/<dataset>.sh`
for multivariate, `scripts/SegRNN/univariate/<dataset>.sh` for univariate
— and reads the metrics they print. Whatever the authors put in those
scripts is what runs, with no interpretation in between.

**Cost.** 32 training runs, roughly 2-3 hours on a T4. Each script covers
one dataset at all four horizons and can be run on its own.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone the authors' original repository, pinned to the last upstream commit.
import os, subprocess

REPO = '/content/SegRNN_pristine'
PRISTINE_COMMIT = '8e869ec'
UPSTREAM = 'https://github.com/lss-1138/SegRNN.git'

if not os.path.isdir(REPO):
    !git clone -q $UPSTREAM $REPO
!cd $REPO && git fetch -q --all && git checkout -q $PRISTINE_COMMIT
print('cloned to', REPO)

In [ ]:
# Verify this really is untouched upstream code before running anything on it.
def sh(cmd):
    return subprocess.run(cmd, shell=True, cwd=REPO, capture_output=True, text=True).stdout.strip()

commit, author = sh('git rev-parse HEAD'), sh('git log -1 --format=%an')
dirty = [l[3:] for l in sh('git status --porcelain').split('\n') if l.strip()]
extra = sorted(f for f in os.listdir(f'{REPO}/models') if f.startswith('SegRNN') and f != 'SegRNN.py')

print('commit          :', commit)
print('author          :', author)
print('date            :', sh('git log -1 --format=%ad --date=short'))
print('modified files  :', dirty or '(none)')
print('extra SegRNN*   :', extra or '(none)')

assert commit.startswith(PRISTINE_COMMIT), f'unexpected commit {commit}'
assert author == 'lss-1138', f'commit not authored upstream: {author}'
assert not extra, f'Stage 2 model files present, not pristine: {extra}'
# utils/tools.py is the one file the next cell is allowed to touch, and only
# for the NumPy 2.0 alias rename documented there. Anything else is a problem.
assert set(dirty) <= {'utils/tools.py'}, f'unexpected modifications: {dirty}'
print('\nOK - upstream code, unmodified except possibly the documented compat patch.')

### One compatibility patch, and why it is safe

The upstream code dates from 2023 and uses `np.Inf`, an alias NumPy
removed in 2.0. Colab now ships NumPy 2.x, so the code raises
`AttributeError` before training starts. There is **exactly one**
occurrence in the whole repository (`utils/tools.py`, the early-stopping
sentinel).

The cell below renames it to `np.inf` and then prints the complete `git
diff` and asserts that it contains precisely that one line. `np.Inf`
*was* `np.inf` — the same object under a deprecated name — so this cannot
change any number produced here. Showing the diff rather than asserting
the claim is the point: the integrity of this notebook rests on the code
being upstream's, so every departure from it is displayed in full.

The alternative is to leave the code byte-identical and pin `numpy<2`
instead. That also works and is left commented out at the bottom of the
cell; it is not the default because downgrading NumPy under Colab's
preinstalled PyTorch build is the less predictable of the two.

In [ ]:
import pathlib

# reset first so this cell is idempotent across re-runs
!cd $REPO && git checkout -q -- utils/tools.py

tools = pathlib.Path(REPO) / 'utils' / 'tools.py'
src = tools.read_text()
assert src.count('np.Inf') == 1, f'expected 1 occurrence, found {src.count("np.Inf")}'
tools.write_text(src.replace('np.Inf', 'np.inf'))

diff = sh('git diff')
print(diff)

changed = [l for l in diff.split('\n')
           if l.startswith(('+', '-')) and not l.startswith(('+++', '---'))]
assert changed == ['-        self.val_loss_min = np.Inf',
                   '+        self.val_loss_min = np.inf'], changed
print('\nOK - exactly one line changed: a removed-alias rename, no numerical effect.')

# Alternative, leaving the code byte-identical (needs a runtime restart):
# !pip install -q "numpy<2"

In [ ]:
# Datasets from Drive, and the dependencies the original repo asks for.
DRIVE_DATASET = '/content/drive/MyDrive/ts-project/dataset'   # same path stage1.ipynb uses

if not os.path.exists(f'{REPO}/dataset'):
    os.symlink(DRIVE_DATASET, f'{REPO}/dataset')
print(sorted(os.listdir(f'{REPO}/dataset')))

!pip install -q -r $REPO/requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## The published values

Read off the paper's Table II (page 6) and Table III (page 7), SegRNN
column. The paper's tables label the second metric column "MSA"; the
values are standard MAE, so they are reported as MAE here.

In [ ]:
import re
import pandas as pd
from IPython.display import display

HORIZONS = [96, 192, 336, 720]
DATASETS = ['ETTh1', 'ETTh2', 'ETTm1', 'ETTm2']

PAPER = {
    'M': {  # Table II, multivariate
        'ETTh1': {'mse': {96: 0.351, 192: 0.392, 336: 0.423, 720: 0.466},
                  'mae': {96: 0.392, 192: 0.414, 336: 0.433, 720: 0.472}},
        'ETTh2': {'mse': {96: 0.276, 192: 0.341, 336: 0.364, 720: 0.403},
                  'mae': {96: 0.335, 192: 0.389, 336: 0.403, 720: 0.448}},
        'ETTm1': {'mse': {96: 0.293, 192: 0.328, 336: 0.357, 720: 0.410},
                  'mae': {96: 0.354, 192: 0.378, 336: 0.399, 720: 0.430}},
        'ETTm2': {'mse': {96: 0.164, 192: 0.226, 336: 0.284, 720: 0.381},
                  'mae': {96: 0.250, 192: 0.294, 336: 0.339, 720: 0.402}},
    },
    'S': {  # Table III, univariate
        'ETTh1': {'mse': {96: 0.053, 192: 0.067, 336: 0.079, 720: 0.078},
                  'mae': {96: 0.180, 192: 0.208, 336: 0.225, 720: 0.224}},
        'ETTh2': {'mse': {96: 0.125, 192: 0.160, 336: 0.186, 720: 0.209},
                  'mae': {96: 0.277, 192: 0.320, 336: 0.350, 720: 0.370}},
        'ETTm1': {'mse': {96: 0.026, 192: 0.040, 336: 0.054, 720: 0.070},
                  'mae': {96: 0.122, 192: 0.154, 336: 0.179, 720: 0.202}},
        'ETTm2': {'mse': {96: 0.063, 192: 0.089, 336: 0.117, 720: 0.156},
                  'mae': {96: 0.183, 192: 0.224, 336: 0.263, 720: 0.309}},
    },
}


def run_script(rel_path):
    """Run one of the repository's own scripts verbatim and collect the
    'mse:..., mae:...' lines it prints, in the order the script emits them
    (each script covers the four horizons in 96/192/336/720 order)."""
    print(f'\n{"="*72}\n$ bash {rel_path}\n{"="*72}')
    proc = subprocess.Popen(['bash', rel_path], cwd=REPO, text=True, bufsize=1,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    found, errors = [], []
    for line in proc.stdout:
        print(line, end='')
        m = re.search(r'mse:([\d.]+), mae:([\d.]+)', line)
        if m:
            found.append((float(m.group(1)), float(m.group(2))))
        elif re.match(r'^\w*(Error|Exception)\b', line.strip()):
            errors.append(line.strip())
    proc.wait()
    if proc.returncode != 0:
        detail = f' -- last error: {errors[-1]}' if errors else ''
        raise RuntimeError(f'{rel_path} exited with {proc.returncode}{detail}')
    if len(found) != len(HORIZONS):
        raise RuntimeError(f'{rel_path}: expected {len(HORIZONS)} results, parsed {len(found)}')
    return dict(zip(HORIZONS, found))


results = {}

## Table II — multivariate

Runs `scripts/SegRNN/etth1.sh`, `etth2.sh`, `ettm1.sh`, `ettm2.sh`
exactly as written. Each covers `pred_len ∈ {96, 192, 336, 720}` at
`seq_len=720`.

In [ ]:
for ds in DATASETS:
    results[(ds, 'M')] = run_script(f'scripts/SegRNN/{ds.lower()}.sh')

print('\nmultivariate done:', sorted(k[0] for k in results if k[1] == 'M'))

## Table III — univariate

Runs `scripts/SegRNN/univariate/etth1.sh` and its siblings, again as
written. These use `--features S` and `--channel_id 0`, matching the
paper's statement that CP encoding is disabled when there is only one
channel.

In [ ]:
for ds in DATASETS:
    results[(ds, 'S')] = run_script(f'scripts/SegRNN/univariate/{ds.lower()}.sh')

print('\nunivariate done:', sorted(k[0] for k in results if k[1] == 'S'))

## Verification

A cell passes if the reconstruction is within **2%** of the published
value on both metrics. `Resolution %` is the finest distinction the
paper's own three-decimal reporting can express (±0.0005 on the
published value) — for the smallest univariate entries that is already
close to 2%, so cells near the threshold there are at the limit of what
the published table can resolve, not necessarily disagreements.

In [ ]:
TOLERANCE_PCT = 2.0
LABEL = {'M': 'II (multivariate)', 'S': 'III (univariate)'}

rows = []
for features in ('M', 'S'):
    for ds in DATASETS:
        for h in HORIZONS:
            mse, mae = results[(ds, features)][h]
            p_mse, p_mae = PAPER[features][ds]['mse'][h], PAPER[features][ds]['mae'][h]
            d_mse, d_mae = (mse / p_mse - 1) * 100, (mae / p_mae - 1) * 100
            rows.append({
                'Table': LABEL[features], 'Dataset': ds, 'Horizon': h,
                'Paper MSE': p_mse, 'Recon MSE': round(mse, 4), 'd MSE %': round(d_mse, 2),
                'Paper MAE': p_mae, 'Recon MAE': round(mae, 4), 'd MAE %': round(d_mae, 2),
                'Resolution %': round(0.0005 / p_mse * 100, 2),
                'Status': 'PASS' if max(abs(d_mse), abs(d_mae)) <= TOLERANCE_PCT else 'FAIL',
            })

df = pd.DataFrame(rows).set_index(['Table', 'Dataset', 'Horizon'])
display(df)
df.to_csv('/content/paper_reconstruction.csv')

n_pass = int((df['Status'] == 'PASS').sum())
print(f'\n{"="*72}')
print(f'{n_pass} of {len(df)} cells within {TOLERANCE_PCT}% of the published values')
print(f'{"="*72}')
for features in ('M', 'S'):
    sub = df.loc[LABEL[features]]
    print(f'  Table {LABEL[features]:20s} mean |d| MSE {sub["d MSE %"].abs().mean():.2f}%   '
          f'mean |d| MAE {sub["d MAE %"].abs().mean():.2f}%   '
          f'signed MSE {sub["d MSE %"].mean():+.2f}%')

failed = df[df['Status'] == 'FAIL']
if len(failed):
    print('\nOutside tolerance:')
    display(failed)
else:
    print('\nAll cells within tolerance.')